Create User embeddings

In [7]:
# ------------------- CREATE eICU PATIENT EMBEDDINGS USING CLINICALBERT -------------------
import pandas as pd
import numpy as np
import re
import os
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import normalize
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# ------------------- Load ClinicalBERT -------------------
print("Loading ClinicalBERT model...")
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
clinicalbert_model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
clinicalbert_model.eval()

# ------------------- Helper Functions -------------------
def encode_sentences(sentences, batch_size=16, max_len=128):
    all_embeddings = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        if not batch:
            continue
        inputs = tokenizer(batch, padding=True, truncation=True, 
                          max_length=max_len, return_tensors="pt")
        with torch.no_grad():
            outputs = clinicalbert_model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_embeddings)
    if not all_embeddings:
        return np.array([])
    return torch.cat(all_embeddings).cpu().numpy()

def aggregate_patient_embedding(note_text):
    if pd.isna(note_text) or not note_text:
        return None
    
    sentences = re.split(r'[.!?]+', note_text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 5]
    
    if len(sentences) == 0:
        sentences = [note_text[:512]]
    
    sentence_embeddings = encode_sentences(sentences)
    if len(sentence_embeddings) == 0:
        return None
    
    sentence_lengths = np.array([len(s) for s in sentences])
    if sentence_lengths.sum() > 0:
        weights = sentence_lengths / sentence_lengths.sum()
        patient_embedding = np.average(sentence_embeddings, axis=0, weights=weights)
    else:
        patient_embedding = np.mean(sentence_embeddings, axis=0)
    
    return normalize(patient_embedding.reshape(1, -1))[0]

# ------------------- Load eICU Data -------------------
eicu_path = r'..........eicu-collaborative-research-database-2.0'

print("\nLoading eICU data...")
patient_eicu = pd.read_csv(os.path.join(eicu_path, 'patient.csv.gz'), low_memory=False)
print(f"patient.csv: {patient_eicu.shape}")

diagnosis_eicu = pd.read_csv(os.path.join(eicu_path, 'diagnosis.csv.gz'), low_memory=False)
print(f"diagnosis.csv: {diagnosis_eicu.shape}")

lab_eicu = pd.read_csv(os.path.join(eicu_path, 'lab.csv.gz'), low_memory=False)
print(f"lab.csv: {lab_eicu.shape}")

admission_eicu = pd.read_csv(os.path.join(eicu_path, 'admissionDx.csv.gz'), low_memory=False)
print(f"admissionDx.csv: {admission_eicu.shape}")

# ------------------- Filter Anemia Patients -------------------
print("\nFiltering anemia patients...")

def extract_icd9_code(icd9_value):
    if pd.isna(icd9_value):
        return None
    icd9_str = str(icd9_value).split(',')[0].split()[0].strip()
    icd9_str = icd9_str.split('.')[0]
    icd9_str = re.sub(r'[^0-9]', '', icd9_str)
    return icd9_str if len(icd9_str) >= 3 else None

anemia_icd9_prefixes = ['280', '281', '282', '283', '284', '285']

diagnosis_eicu['ICD9_CODE'] = diagnosis_eicu['icd9code'].apply(extract_icd9_code)
diagnosis_eicu = diagnosis_eicu.dropna(subset=['ICD9_CODE'])

diagnoses_anemia = diagnosis_eicu[
    diagnosis_eicu['ICD9_CODE'].str[:3].isin(anemia_icd9_prefixes)
]
anemia_patient_ids = set(diagnoses_anemia['patientunitstayid'].unique())
print(f"Anemia patients: {len(anemia_patient_ids)}")

# ------------------- Process Lab Data -------------------
print("\nProcessing lab data...")

relevant_labs = [
    'hemoglobin', 'hgb', 'hematocrit', 'hct', 'rbc', 'red blood cell',
    'mcv', 'mean corpuscular volume', 'ferritin', 'iron', 'transferrin',
    'tibc', 'b12', 'vitamin b12', 'folate', 'reticulocyte', 'bilirubin',
    'ldh', 'haptoglobin', 'creatinine', 'egfr'
]

labs_by_patient = {}
for _, row in tqdm(lab_eicu.iterrows(), total=len(lab_eicu), desc="Processing labs"):
    pid = row['patientunitstayid']
    lab_name = str(row.get('labname', row.get('result', ''))).lower()
    
    is_relevant = any(relevant in lab_name for relevant in relevant_labs)
    
    if is_relevant:
        if pid not in labs_by_patient:
            labs_by_patient[pid] = []
        
        lab_value = row.get('labresult', row.get('result', row.get('value', None)))
        if pd.notna(lab_value):
            try:
                value = float(lab_value)
                labs_by_patient[pid].append(value)
            except (ValueError, TypeError):
                pass

print(f"  Patients with lab data: {len(labs_by_patient)}")

# ------------------- Build Clinical Text for Each Patient -------------------
print("\nBuilding clinical text...")

patient_dict = {row['patientunitstayid']: row for _, row in patient_eicu.iterrows()}
admission_dict = {row['patientunitstayid']: row for _, row in admission_eicu.iterrows()}

diagnoses_by_patient = {}
for _, row in diagnosis_eicu.iterrows():
    pid = row['patientunitstayid']
    if pid not in diagnoses_by_patient:
        diagnoses_by_patient[pid] = []
    diag = row.get('diagnosisstring', row.get('icd9code', ''))
    if pd.notna(diag):
        diagnoses_by_patient[pid].append(str(diag))



def build_clinical_text(patient_id):
    text_parts = []
    
    patient_row = patient_dict.get(patient_id)
    if patient_row is not None:
        age = patient_row.get('age', '')
        gender = patient_row.get('gender', '')
        if pd.notna(age) and age:
            text_parts.append(f"Age: {age}")
        if pd.notna(gender) and gender:
            text_parts.append(f"Gender: {gender}")
    
    adm_row = admission_dict.get(patient_id)
    if adm_row is not None:
        admit_dx = adm_row.get('admitdx', adm_row.get('admitdxname', adm_row.get('diagnosis', '')))
        if pd.notna(admit_dx) and admit_dx:
            text_parts.append(f"Admission Diagnosis: {admit_dx}")
    
    diagnoses = diagnoses_by_patient.get(patient_id, [])
    if diagnoses:
        text_parts.append(f"Diagnoses: {'; '.join(diagnoses[:10])}")
    
    labs = labs_by_patient.get(patient_id, [])
    if labs:
        lab_summary = ', '.join([f"lab_{i}:{v:.1f}" for i, v in enumerate(labs[:20])])
        text_parts.append(f"Labs: {lab_summary}")
    
    if not text_parts:
        return None
    
    return '\n'.join(text_parts)

# ------------------- Generate Embeddings -------------------
print("\nGenerating patient embeddings...")

patient_embeddings = []
patient_ids = []

for patient_id in tqdm(anemia_patient_ids, desc="Processing"):
    clinical_text = build_clinical_text(patient_id)
    
    if clinical_text and len(clinical_text) > 10:
        embedding = aggregate_patient_embedding(clinical_text)
        if embedding is not None:
            patient_embeddings.append(embedding)
            patient_ids.append(str(patient_id))

# ------------------- Save Embeddings -------------------
if patient_embeddings:
    eicu_patient_emb_matrix = np.vstack(patient_embeddings)
    eicu_patient_id_to_idx = {pid: idx for idx, pid in enumerate(patient_ids)}
    
    output_dir = r'.................'
    np.save(os.path.join(output_dir, 'eicu_patient_embeddings.npy'), eicu_patient_emb_matrix)
    
    with open(os.path.join(output_dir, 'eicu_patient_id_to_idx.json'), 'w') as f:
        json.dump(eicu_patient_id_to_idx, f)
    
    print(f"\n✓ Saved {len(patient_ids)} embeddings (dim={eicu_patient_emb_matrix.shape[1]})")
    print(f"  Success rate: {len(patient_ids)}/{len(anemia_patient_ids)} ({100*len(patient_ids)/len(anemia_patient_ids):.1f}%)")
    
    print(f"\nEmbedding statistics:")
    print(f"  Mean: {eicu_patient_emb_matrix.mean():.6f}")
    print(f"  Std: {eicu_patient_emb_matrix.std():.6f}")
    print(f"  Min: {eicu_patient_emb_matrix.min():.6f}")
    print(f"  Max: {eicu_patient_emb_matrix.max():.6f}")
else:
    print("No valid embeddings generated!")

print("\n✅ eICU PATIENT EMBEDDINGS CREATED")
print(f"   Final coverage: {len(patient_ids)}/{len(anemia_patient_ids)} patients (100%)")
print(f"   Embedding dimension: {eicu_patient_emb_matrix.shape[1]}")

Loading ClinicalBERT model...

Loading eICU data...
patient.csv: (200859, 29)
diagnosis.csv: (2710672, 7)
lab.csv: (39132531, 10)
admissionDx.csv: (626858, 6)

Filtering anemia patients...
Anemia patients: 5198

Processing lab data...


Processing labs: 100%|██████████████████████████████████████████████████| 39132531/39132531 [40:15<00:00, 16201.00it/s]


  Patients with lab data: 192469

Building clinical text...

Generating patient embeddings...


Processing: 100%|████████████████████████████████████████████████████████████████| 5198/5198 [1:07:15<00:00,  1.29it/s]



✓ Saved 5198 embeddings (dim=768)
  Success rate: 5198/5198 (100.0%)

Embedding statistics:
  Mean: -0.000503
  Std: 0.036081
  Min: -0.864308
  Max: 0.090479

✅ eICU PATIENT EMBEDDINGS CREATED
   Final coverage: 5198/5198 patients (100%)
   Embedding dimension: 768
